# Actividad (Parte 5) Explicabilidad usando contraejemplos
## Materia: Inteligencia Artificial Explicable - MSc. Inteligencia Artificial
Author: Esteban García-Cuesta, Departamento de Inteligencia Artificial, UPM (License CC-BY-NC)

This code has been developed to be used exclusively for educational purposes.

## Introducción
La explicabilidad con contraejemplos permite conocer el funcionamiento del modelo modificando las entradas e identificar las opciones posibles para revertir una decisión no favorable. En esta actividad utilizaras la base de datos Heart Disease UCI Machine Learning Repository.

## Objetivos:
  - Aprender como obtener un contraejemplo dado un caso
  - Aprender a obtener un mejor contraejemplo usando el parámetro $\lambda$ de la fórmula de Watcher $\lambda (f_{\theta}(x')-y')+d(x^e-x')$
  - Aprender a aplicar en un caso de reversión de decisión las técnicas de contraejemplo e interpretar los resultados

## Para hacer
  - Realiza los cambios en el código necesarios de acuerdo a las instrucciones de la actividad y responde a las preguntas que se indican en dichas instrucciones.

In [1]:
# Instalación de la librerio ucimlrepo
!pip install ucimlrepo

In [2]:
#Lectura de los datos desde el repositorio UCI
from ucimlrepo import fetch_ucirepo
import pandas as pd

# fetch dataset
heart_disease = fetch_ucirepo(id=45)

# data (as pandas dataframes)
dfX = heart_disease.data.features
dfY = heart_disease.data.targets
df = pd.concat([dfX, dfY], axis=1)
df = df.dropna()
dfX = df[['age','trestbps','chol','thalach','oldpeak','ca']]
dfY = df['num']
dfX.head()


,age,trestbps,chol,thalach,oldpeak,ca
0,63,145,233,150,2.3,0.0
1,67,160,286,108,1.5,3.0
2,67,120,229,129,2.6,2.0
3,37,130,250,187,3.5,0.0
4,41,130,204,172,1.4,0.0


In [3]:
# Aprendizaje del modelo random forest

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import numpy as np

X = dfX.values
y = np.ravel(dfY.values)

clf = RandomForestClassifier(random_state=0)
clf.fit(X, y)

RandomForestClassifier(random_state=0)

In [4]:
# Predicción para el ejemplo 55
from mlxtend.evaluate import create_counterfactual

x_ref = X[55]

print('True label:', y[55])
print('Predicted label:', clf.predict(x_ref.reshape(1, -1))[0])
print('Predicted probas:', clf.predict_proba(x_ref.reshape(1, -1)))

True label: 1
Predicted label: 1
Predicted probas: [[0.02 0.72 0.07 0.19 0.  ]]


In [6]:
# Búsqueda de un contraejemplo

res = create_counterfactual(x_reference=x_ref,
                            y_desired=0,
                            model=clf,
                            X_dataset=X,
                            y_desired_proba=1,
                            lammbda=1, #  hyperparameter
                            random_seed=123)

print('Features of the 55th example:', x_ref)
print('Features of the countefactual:', res)

print('Predictions for counterfactual:\n')
print('Predicted label:', clf.predict(res.reshape(1, -1))[0])
print('Predicted probas:', clf.predict_proba(res.reshape(1, -1)))

Features of the 55th example: [ 54.  124.  266.  109.    2.2   1. ]
Features of the countefactual: [ 5.39803423e+01  1.23999991e+02  2.65999967e+02  1.09500107e+02
 -3.92200090e-04  9.64418746e-04]
Predictions for counterfactual:

Predicted label: 0
Predicted probas: [[0.57 0.25 0.05 0.13 0.  ]]
